# 03 · Qwen2.5-VL-7B QLoRA (Unsloth)
Güçlü ray: `Qwen2.5-VL-7B-Instruct` (Apache-2.0) görsel-dil modeli, kutu düzeyinde tüm cevabı okur — satır bölütleme gerektirmez. Unsloth ile ücretsiz T4'te QLoRA. Eğitim sonrası Mac'te **mlx-vlm** ile çalıştırılır.

> 3B sürümü ticari değildir; 7B/Apache tercih edin. Qwen3-VL 4B/8B (Apache) el yazısında daha iyidir, çıktı da hafifler — alternatif olarak deneyin.

In [ ]:
!pip install -q unsloth
from google.colab import drive; drive.mount('/content/drive')
!git clone https://github.com/KULLANICI/sinav-okur.git  # <-- kendi repo URL'nizle değiştirin || true
%cd sinav-okur && pip install -q -e . && cd ..

In [ ]:
!unzip -qo /content/drive/MyDrive/sinav_okur_veri.zip -d /content
import json, os
kayitlar = [json.loads(l) for l in open('/content/veri/labels.jsonl')]
for k in kayitlar: k['image'] = os.path.join('/content/veri', k['image'])

In [ ]:
from unsloth import FastVisionModel
import torch
model, islemci = FastVisionModel.from_pretrained(
    'unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit',
    load_in_4bit=True, use_gradient_checkpointing='unsloth')
model = FastVisionModel.get_peft_model(model, r=16, lora_alpha=16,
    finetune_vision_layers=True, finetune_language_layers=True)

In [ ]:
KOMUT = ('Bu görüntüde Türkçe el yazısıyla yazılmış bir sınav cevabı var. '
         'Metni AYNEN yaz, düzeltme yapma.')
from PIL import Image
def ornekle(k):
    return {'messages': [
        {'role':'user','content':[{'type':'image'},{'type':'text','text':KOMUT}]},
        {'role':'assistant','content':[{'type':'text','text':k['text']}]}]}
from datasets import Dataset
ds = Dataset.from_list([{**k, **ornekle(k)} for k in kayitlar])

In [ ]:
from unsloth import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
FastVisionModel.for_training(model)
trainer = SFTTrainer(model=model, tokenizer=islemci,
    data_collator=UnslothVisionDataCollator(model, islemci),
    train_dataset=ds,
    args=SFTConfig(per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=10, num_train_epochs=1, learning_rate=2e-4, fp16=True,
        logging_steps=25, output_dir='qwen-vl-tr', report_to='none',
        remove_unused_columns=False, dataset_kwargs={'skip_prepare_dataset': True},
        max_seq_length=1024))
trainer.train()

In [ ]:
# LoRA'yı birleştirip 16-bit kaydet, Drive'a at
model.save_pretrained_merged('qwen-vl-tr-merged', islemci)
!zip -qr /content/drive/MyDrive/qwen-vl-tr.zip qwen-vl-tr-merged

## Mac'te mlx'e çevir
İndirdikten sonra Mac terminalinde:
```bash
pip install mlx-vlm
python -m mlx_vlm.convert --hf-path ./qwen-vl-tr-merged -q --mlx-path ./qwen-vl-tr-mlx
sinav-okur isle ... --taniyici mlx --model ./qwen-vl-tr-mlx
```